In [1]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "622":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))

        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora"
SPLIT_TYPE = "fixed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 16:48:20,196] A new study created in memory with name: no-name-1fb41cde-9044-440c-813e-ec83003add5a



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 16:57:08,333] Trial 0 finished with value: 0.8100000421206156 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8100000421206156.
[I 2026-09-21 16:57:15,304] Trial 1 finished with value: 0.7986666957537333 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.8100000421206156.
[I 2026-09-21 16:57:17,647] Trial 2 finished with value: 0.8126667141914368 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.8126667141914368.
[I 2026-09-21 16:57:20,880] Trial 3 finished with value: 0.8020000457763672 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.8126

[I 2026-09-21 17:00:13,982] A new study created in memory with name: no-name-6e4d9d7c-374f-4fb3-a35b-a7f2b89c0049


GCNII: 0.8237 +/- 0.0085

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 17:00:22,050] Trial 0 finished with value: 0.8126667141914368 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8126667141914368.
[I 2026-09-21 17:00:33,298] Trial 1 finished with value: 0.8166666825612386 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.8166666825612386.
[I 2026-09-21 17:00:40,803] Trial 2 finished with value: 0.8093333840370178 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.8166666825612386.
[I 2026-09-21 17:00:46,957] Trial 3 finished with value: 0.802666703859965 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8166666825612

In [2]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 17:05:47,207] A new study created in memory with name: no-name-8441d7aa-253a-4611-a7f4-c01d2d8c2089



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 17:14:16,857] Trial 0 finished with value: 0.898524006207784 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.898524006207784.
[I 2026-09-21 17:14:27,749] Trial 1 finished with value: 0.898524006207784 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.898524006207784.
[I 2026-09-21 17:14:31,053] Trial 2 finished with value: 0.9059040546417236 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.9059040546417236.
[I 2026-09-21 17:14:35,834] Trial 3 finished with value: 0.8936039606730143 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.90590405

[I 2026-09-21 17:19:56,875] A new study created in memory with name: no-name-7149ed85-7489-428c-b000-f000c70c84f1


GCNII: 0.8928 +/- 0.0133

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 17:20:03,498] Trial 0 finished with value: 0.9015990296999613 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9015990296999613.
[I 2026-09-21 17:20:12,980] Trial 1 finished with value: 0.9015990296999613 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9015990296999613.
[I 2026-09-21 17:20:19,098] Trial 2 finished with value: 0.9059040745099386 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.9059040745099386.
[I 2026-09-21 17:20:23,716] Trial 3 finished with value: 0.8954489429791769 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.905904074509

In [3]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 17:23:19,812] A new study created in memory with name: no-name-d47b2d6b-9413-4ba7-951b-16698bc48a86



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 17:36:27,261] Trial 0 finished with value: 0.7779448628425598 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.7779448628425598.
[I 2026-09-21 17:36:35,544] Trial 1 finished with value: 0.7759398420651754 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.7779448628425598.
[I 2026-09-21 17:36:38,571] Trial 2 finished with value: 0.7804511586825053 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.7804511586825053.
[I 2026-09-21 17:36:44,489] Trial 3 finished with value: 0.7794486284255981 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.7804

[I 2026-09-21 17:40:23,390] A new study created in memory with name: no-name-a8b52778-c15b-4eae-a6c6-ea371939cff1


GCNII: 0.7701 +/- 0.0194

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 17:40:34,655] Trial 0 finished with value: 0.7729323307673136 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7729323307673136.
[I 2026-09-21 17:40:52,518] Trial 1 finished with value: 0.7739348411560059 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.7739348411560059.
[I 2026-09-21 17:41:04,878] Trial 2 finished with value: 0.7809523940086365 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.7809523940086365.
[I 2026-09-21 17:41:13,761] Trial 3 finished with value: 0.774937371412913 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.7809523940086

In [4]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "PubMed"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 17:47:01,411] A new study created in memory with name: no-name-16684194-09ac-4f8c-be83-608449b2f527



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 18:37:07,558] Trial 0 finished with value: 0.8607659538586935 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8607659538586935.
[I 2026-09-21 18:39:41,690] Trial 1 finished with value: 0.8874799410502116 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8874799410502116.
[I 2026-09-21 18:40:17,476] Trial 2 finished with value: 0.8767436345418295 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.8874799410502116.
[I 2026-09-21 18:40:38,304] Trial 3 finished with value: 0.861864964167277 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.88747

[I 2026-09-21 19:31:11,540] A new study created in memory with name: no-name-6deda235-e653-455c-b496-46dbad906392


GCNII: 0.8880 +/- 0.0054

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 19:31:54,319] Trial 0 finished with value: 0.8672753771146139 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8672753771146139.
[I 2026-09-21 19:33:36,414] Trial 1 finished with value: 0.8686279853185018 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.8686279853185018.
[I 2026-09-21 19:33:58,507] Trial 2 finished with value: 0.8627948562304179 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.8686279853185018.
[I 2026-09-21 19:34:36,751] Trial 3 finished with value: 0.9034576416015625 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.903457641601

In [5]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "622":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))

        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "fixed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

Optuna search for GCN


[I 2026-09-21 20:01:11,730] A new study created in memory with name: no-name-6d02b1dc-4bff-4f91-b62f-e904e8a2e4e8


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


[I 2026-09-21 20:01:27,877] Trial 0 finished with value: 0.7000000278155009 and parameters: {'hidden': 32, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.7000000278155009.
[I 2026-09-21 20:01:30,775] Trial 1 finished with value: 0.7000000476837158 and parameters: {'hidden': 16, 'dropout': 0.5, 'lr': 0.005, 'weight_decay': 0.005}. Best is trial 1 with value: 0.7000000476837158.
[I 2026-09-21 20:01:32,348] Trial 2 finished with value: 0.7106666962305704 and parameters: {'hidden': 64, 'dropout': 0.5, 'lr': 0.01, 'weight_decay': 0.005}. Best is trial 2 with value: 0.7106666962305704.
[I 2026-09-21 20:01:33,608] Trial 3 finished with value: 0.7073333660761515 and parameters: {'hidden': 64, 'dropout': 0.5, 'lr': 0.02, 'weight_decay': 0.001}. Best is trial 2 with value: 0.7106666962305704.
[I 2026-09-21 20:01:35,169] Trial 4 finished with value: 0.6820000410079956 and parameters: {'hidden': 16, 'dropout': 0.3, 'lr': 0.01, 'weight_decay': 0.0005}. Best is tr

...

[I 2026-09-21 20:15:51,130] Trial 0 finished with value: 0.7140000263849894 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.7140000263849894.
[I 2026-09-21 20:16:06,256] Trial 1 finished with value: 0.7280000249544779 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.7280000249544779.
[I 2026-09-21 20:16:10,030] Trial 2 finished with value: 0.7226667006810507 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.7280000249544779.
[I 2026-09-21 20:16:13,068] Trial 3 finished with value: 0.7200000286102295 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.7280

[I 2026-09-21 20:21:08,544] A new study created in memory with name: no-name-bc2230ac-4b78-4a04-8fd9-e96839d5a27f


GCNII: 0.7045 +/- 0.0055

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 20:21:19,706] Trial 0 finished with value: 0.7286667029062907 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7286667029062907.
[I 2026-09-21 20:21:36,584] Trial 1 finished with value: 0.7193333903948466 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7286667029062907.
[I 2026-09-21 20:21:48,543] Trial 2 finished with value: 0.7200000484784445 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7286667029062907.
[I 2026-09-21 20:22:08,127] Trial 3 finished with value: 0.718000054359436 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.7286667029062

In [6]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Planetoid(root=root, name=dataset_name)
    elif split_type == "622":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Planetoid(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))

        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "PubMed"
SPLIT_TYPE = "fixed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 20:31:08,880] A new study created in memory with name: no-name-df9d328f-028d-4040-ac80-21fcf549a12c



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 20:53:22,462] Trial 0 finished with value: 0.812000056107839 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.812000056107839.
[I 2026-09-21 20:54:24,679] Trial 1 finished with value: 0.8146666884422302 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.8146666884422302.
[I 2026-09-21 20:54:36,345] Trial 2 finished with value: 0.815333366394043 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.815333366394043.
[I 2026-09-21 20:54:44,820] Trial 3 finished with value: 0.812000036239624 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.815333366

[I 2026-09-21 21:21:03,290] A new study created in memory with name: no-name-0424f66d-6ed3-4166-9214-1df05f608355


GCNII: 0.7974 +/- 0.0034

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 21:21:27,319] Trial 0 finished with value: 0.8253333767255148 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8253333767255148.
[I 2026-09-21 21:21:56,880] Trial 1 finished with value: 0.8046666979789734 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8253333767255148.
[I 2026-09-21 21:22:15,772] Trial 2 finished with value: 0.8186666965484619 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8253333767255148.
[I 2026-09-21 21:22:41,056] Trial 3 finished with value: 0.8053333759307861 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.825333376725

In [7]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "CiteSeer"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 21:35:39,874] A new study created in memory with name: no-name-2fd8b231-6a7d-4d3b-a8a6-e81ed8adacca



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 21:44:29,145] Trial 0 finished with value: 0.9389282862345377 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9389282862345377.
[I 2026-09-21 21:44:43,533] Trial 1 finished with value: 0.9479905366897583 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9479905366897583.
[I 2026-09-21 21:44:47,347] Trial 2 finished with value: 0.9428683916727701 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.9479905366897583.
[I 2026-09-21 21:44:54,903] Trial 3 finished with value: 0.9369582335154215 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9479

[I 2026-09-21 21:50:42,335] A new study created in memory with name: no-name-9cadadff-1f34-421c-b6bd-cc0aa24ed528


GCNII: 0.9485 +/- 0.0030

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 21:50:46,899] Trial 0 finished with value: 0.9452324708302816 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9452324708302816.
[I 2026-09-21 21:50:56,076] Trial 1 finished with value: 0.9487785696983337 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9487785696983337.
[I 2026-09-21 21:51:00,366] Trial 2 finished with value: 0.9381402532259623 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9487785696983337.
[I 2026-09-21 21:51:05,249] Trial 3 finished with value: 0.9539007147153219 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.953900714715

In [8]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import CitationFull
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = CitationFull(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = CitationFull(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cora_ML"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 21:54:29,301] A new study created in memory with name: no-name-8ae0741a-a049-407a-af32-4b7ca4917779



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-21 22:13:53,841] Trial 0 finished with value: 0.8953811724980673 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.8953811724980673.
[I 2026-09-21 22:14:04,907] Trial 1 finished with value: 0.8953811724980673 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.8953811724980673.
[I 2026-09-21 22:14:07,531] Trial 2 finished with value: 0.8964941302935282 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.8964941302935282.
[I 2026-09-21 22:14:11,523] Trial 3 finished with value: 0.8925987283388773 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.8964

[I 2026-09-21 22:17:58,005] A new study created in memory with name: no-name-1859069d-49e7-45b7-8c77-d7894d5de30e


GCNII: 0.8925 +/- 0.0094

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-21 22:18:24,192] Trial 0 finished with value: 0.9048413832982382 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9048413832982382.
[I 2026-09-21 22:18:46,829] Trial 1 finished with value: 0.9009459813435873 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9048413832982382.
[I 2026-09-21 22:19:01,285] Trial 2 finished with value: 0.8992765545845032 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9048413832982382.
[I 2026-09-21 22:19:21,351] Trial 3 finished with value: 0.8964941501617432 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.904841383298

In [9]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import Amazon
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = Amazon(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = Amazon(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Photo"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-21 22:29:27,484] A new study created in memory with name: no-name-3bf6c77b-9ac0-4c7b-b038-0c699d9b6584



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 00:31:51,838] Trial 0 finished with value: 0.9494553605715433 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.9494553605715433.
[I 2026-09-22 00:36:10,127] Trial 1 finished with value: 0.9596950014432272 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9596950014432272.
[I 2026-09-22 00:37:00,608] Trial 2 finished with value: 0.9542483687400818 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 1 with value: 0.9596950014432272.
[I 2026-09-22 00:38:02,460] Trial 3 finished with value: 0.9457516471544901 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.9596

[I 2026-09-22 01:51:28,026] A new study created in memory with name: no-name-5faa4d6b-b083-4aeb-b648-5e4680a7bd3b


GCNII: 0.9518 +/- 0.0037

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 01:52:45,260] Trial 0 finished with value: 0.9503268003463745 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.9503268003463745.
[I 2026-09-22 01:55:27,062] Trial 1 finished with value: 0.9529411991437277 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9529411991437277.
[I 2026-09-22 01:56:43,733] Trial 2 finished with value: 0.9507625500361124 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.9529411991437277.
[I 2026-09-22 01:58:07,289] Trial 3 finished with value: 0.9555555780728658 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.955555578072